# 第4章 無線チャネルの表現

書籍『Sionna RTによる電波伝搬レイトレーシング入門』第4章のコードである。
セルは本文の掲載順に並べてあり、コードセルの中身は本文に印刷したものと同じである。

先頭の「準備」セルだけは本文に印刷していない。本文のコードが前提にしている
import とシーンの読み込みをここで済ませてある。準備セルを実行すれば、以降は
上から順に実行できる。

ただし、本文が説明のために示している断片（自分で作ったシーンのパスや、その
シーンには無い物体名を使う例）はそのままでは動かない。該当するセルの前に注記を
置いてある。

## 準備

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

import sionna.rt as rt
from sionna.rt import (load_scene, scene as scene_assets,
                       Transmitter, Receiver, PlanarArray,
                       PathSolver, RadioMapSolver, Camera, RadioMaterial)

# 第3章・第4章で使う見通しの効く街路シーン
scene = load_scene(scene_assets.simple_street_canyon)
scene.frequency = 3.5e9
scene.tx_array = PlanarArray(num_rows=1, num_cols=1,
                             pattern='iso', polarization='V')
scene.rx_array = PlanarArray(num_rows=1, num_cols=1,
                             pattern='iso', polarization='V')
scene.add(Transmitter('tx0', position=[-70.0, -6.0, 10.0]))
scene.add(Receiver('rx0', position=[70.0, 6.0, 1.5]))

solver = PathSolver()
paths = solver(scene, max_depth=5)

## 4. チャネルインパルス応答（CIR）

In [ ]:
# パスから CIR を NumPy 配列として取り出す
h, tau = paths.cir(normalize_delays=True, out_type='numpy')
print('h shape:', h.shape, 'tau shape:', tau.shape)

## 6. チャネル周波数応答（CFR）

In [ ]:
import drjit as dr
import mitsuba as mi

# 搬送波に対する周波数オフセット [-50, 50] MHz
freq_offsets = dr.linspace(mi.Float, -50e6, 50e6, 1024)
H = paths.cfr(freq_offsets, normalize_delays=True,
              out_type='numpy')

## 8. チャネルタップ

In [ ]:
# 帯域幅 100 MHz、タップ番号 0〜63 を生成
taps = paths.taps(bandwidth=100e6, l_min=0, l_max=63,
                  out_type='numpy')
print('taps shape:', taps.shape)

## 10. 遅延スプレッドを計算する

In [ ]:
def rms_delay_spread(a, tau):
    p = (abs(a)**2).flatten()
    t = tau.flatten()
    mean_tau = (p*t).sum() / p.sum()
    return np.sqrt((p*(t-mean_tau)**2).sum() / p.sum())

print(f'RMS delay spread: {rms_delay_spread(h, tau)*1e9:.2f} ns')

## 11. 角度情報を読み取る

In [ ]:
# 有効なパスの到来角
valid = np.array(paths.valid).flatten()
phi_r = np.array(paths.phi_r).flatten()[valid]
theta_r = np.array(paths.theta_r).flatten()[valid]
print('theta [deg]:', np.degrees(theta_r))
print('phi [deg]:', np.degrees(phi_r))

## 12. NumPy や PyTorch に出力する

In [ ]:
a_np, tau_np = paths.cir(out_type='numpy')
a_torch, tau_torch = paths.cir(out_type='torch')